# MicroDuck training on Google Colab
Open this notebook in Colab, choose a GPU runtime, then run the cell. GPU access and compute-unit use depend on your account.
在 Colab 中打开此 Notebook，选择 GPU 运行时并运行代码。可用 GPU 与算力消耗由您的账号决定。

In [ ]:
import os, subprocess, pathlib, shutil
task = 'Mjlab-Velocity-Flat-MicroDuck'
iterations = 1000
envs = 64
repo = pathlib.Path('/content/microduck_rl')
if not shutil.which('uv'):
    subprocess.run(['python', '-m', 'pip', 'install', 'uv'], check=True)
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/pollen-robotics/microduck_rl.git', str(repo)], check=True)
subprocess.run(['uv', 'sync', '--no-dev'], cwd=repo, check=True)
subprocess.run(['uv', 'run', 'train', task, '--env.scene.num-envs', str(envs), '--agent.max_iterations', str(iterations)], cwd=repo, env={**os.environ, 'WANDB_MODE':'disabled'}, check=True)
checkpoints = sorted(repo.glob('logs/**/model_*.pt'), key=lambda p: p.stat().st_mtime)
assert checkpoints, 'No checkpoint was produced'
subprocess.run(['uv','run','python','scripts/export.py',task,'--checkpoint-file',str(checkpoints[-1]),'--num-envs','1','--onnx-file','/content/policy.onnx'], cwd=repo, check=True)
from google.colab import files
files.download('/content/policy.onnx')
